# Adding the "filename" of fits files to a new list of TOIs for which you have a .csv file

In [1]:
#Read the name of all the files in /pdo/users/pablomer/mnt/tess/astronet/new_fits_files_list.txt

import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.io import fits


In [ ]:
#Read the file with the list of files
# file_list = '/pdo/users/pablomer/mnt/tess/astronet/new_fits_files_list.txt'
file_list = '/pdo/users/pablomer/mnt/tess/astronet/junk_examples_fits_files_list.txt'

with open(file_list) as f:
    files = f.readlines()
files = [x.strip() for x in files]

#Load the .csv file from /pdo/users/pablomer/mnt/tess/labels/vetting-new-events.csv
new_events = pd.read_csv('/pdo/users/pablomer/mnt/tess/labels/vetting-new-events.csv')

In [7]:
print(len(files),'fits files')
print(new_events.shape[0],'new events')

1580 fits files
1647 new events


In [ ]:
# print(new_events.columns)

Index(['Parameter Source Pipeline', 'Detection Pipeline(s)', 'TIC ID',
       'Full TOI ID', 'Final', 'RA', 'Dec', 'TIC Right Ascension 2015.5',
       'TIC Declination 2015.5', 'Tmag', 'TMag Uncertainty', 'Epoc',
       'Epoch Uncertainty', 'Period', 'Orbital Period Uncertainty', 'Duration',
       'Transit Duration (hours) Uncertainty', 'Transit_Depth',
       'Transit Depth Uncertainty', 'Sectors', 'comment', 'logg',
       'Surface Gravity Uncertainty', 'Slug', 'star_rad',
       'Star Radius Uncertainty', 'Planet Radius Value',
       'Planet Radius Uncertainty', 'Planet Equilibrium Temperature (K) Value',
       'teff', 'Effective Temperature Uncertainty',
       'Effective Stellar Flux Value', 'SN', 'Centroid Offset', 'Master',
       'SG1a', 'SG1b', 'SG2', 'SG3', 'SG4', 'SG5', 'Alerted', 'Updated',
       'Astro ID', 'Decision', 'Distinct', 'mk', 'ch', 'et', 'md', 'as', 'dm',
       'Tansu', 'Shishir', 'jh', 'Astronet note', 'Seed randbetween(1, 100)',
       'star_rad_est', 'f

In [ ]:
# new_events['TIC ID'].shape

(1647,)

In [10]:
# new_events['Full TOI ID']

0        120.01
1        127.01
2        137.01
3        142.01
4        145.01
         ...   
1642    7181.01
1643    7182.01
1644    7183.01
1645    7184.01
1646    7185.01
Name: Full TOI ID, Length: 1647, dtype: float64

In [9]:
new_events['filename'] = new_events['filename'].astype(str)

In [47]:
#Remove duplicate TIC IDs from the new_events dataframe
# NOT NECESSARY! As long as they have unique astroIDs, we can have repeated TIC IDs
# new_events_no_dpl = new_events.drop_duplicates(subset='TIC ID')

In [48]:
# # Match the TIC ID of new_events with the name of the files


# num_matches = 0
# for i in range(new_events_no_dpl['TIC ID'].shape[0]):
#     tic_id = new_events_no_dpl['TIC ID'].iloc[i]
#     tic_id = str(tic_id)
#     tic_id = tic_id.zfill(len('0000000001167538'))
#     # print(tic_id)
#     for j in range(len(files)):
#         if tic_id in files[j]:
#             #print('Found a match for', files[j])
#             num_matches = num_matches + 1
#             new_events_no_dpl.loc[new_events_no_dpl['TIC ID'] == new_events_no_dpl['TIC ID'].iloc[i], 'filename'] = files[j]

# print('Number of matches:', num_matches, 'out of', len(new_events_no_dpl))
# print('The length of the list of files is:', len(files))


In [11]:
# Match the TIC ID of new_events with the name of the files
num_matches = 0
for i in range(new_events['TIC ID'].shape[0]):
    tic_id = new_events['TIC ID'].iloc[i]
    tic_id = str(tic_id)
    tic_id = tic_id.zfill(len('0000000001167538'))
    # print(tic_id)
    for j in range(len(files)):
        if tic_id in files[j]:
            #print('Found a match for', files[j])
            num_matches = num_matches + 1
            new_events.loc[new_events['TIC ID'] == new_events['TIC ID'].iloc[i], 'filename'] = files[j]

print('Number of matches:', num_matches, 'out of', len(new_events))
print('The length of the list of files is:', len(files))

Number of matches: 1647 out of 1647
The length of the list of files is: 1580


In [12]:
print(new_events[['filename', 'TIC ID','Astro ID']].head())


                                            filename     TIC ID  Astro ID
0  astronet_hlsp_qlp_tess_ffi-s0085-0000000394137...  394137592      9345
1  astronet_hlsp_qlp_tess_ffi-s0085-0000000234523...  234523599      9346
2  astronet_hlsp_qlp_tess_ffi-s0085-0000000176957...  176957796      9347
3  astronet_hlsp_qlp_tess_ffi-s0085-0000000425934...  425934411      9348
4  astronet_hlsp_qlp_tess_ffi-s0085-0000000265612...  265612438      9349


In [13]:
# Save new_events to mnt/tess/labels/vetting-new-events-withfilenames.csv

new_events.to_csv('/pdo/users/pablomer/mnt/tess/labels/vetting-new-events-withfilenames.csv', index=False)

Other duplicate-checks

In [51]:
# Check if there is any repeated TIC IDs in new_events
tic_ids = new_events['TIC ID']
# tic_ids = tic_ids.astype(str)
# tic_ids = tic_ids.str.zfill(len('0000000001167538'))
tic_ids = tic_ids.drop_duplicates()
print('Number of unique TIC IDs:', len(tic_ids))
print('Number of repeated TIC IDs:', len(new_events) - len(tic_ids))


Number of unique TIC IDs: 1580
Number of repeated TIC IDs: 67


In [52]:
# Count how many times each TIC ID appears
duplicate_counts = new_events['TIC ID'].value_counts()

# Get only the duplicate TIC IDs (i.e., those appearing more than once)
num_duplicates = (duplicate_counts > 1).sum()

print(f"Number of duplicate TIC IDs: {num_duplicates}")
duplicate_tic_ids = duplicate_counts[duplicate_counts > 1].index.tolist()
print("Duplicate TIC IDs:", duplicate_tic_ids)
total_duplicate_count = new_events.duplicated(subset=['TIC ID'], keep=False).sum()
print(f"Total number of duplicate occurrences: {total_duplicate_count}")


Number of duplicate TIC IDs: 53
Duplicate TIC IDs: [260647166, 64837857, 393546540, 459837008, 136916387, 280031353, 27491137, 257605131, 207425167, 441798995, 308994098, 161477033, 80224448, 230387153, 352239069, 347332255, 404518509, 17129975, 54002556, 31852980, 396720998, 153949511, 219175972, 431810418, 119973835, 149989864, 391903064, 304690618, 349972412, 234388232, 443666343, 349488688, 455947620, 426032475, 262715204, 443616612, 143022742, 198153540, 127505658, 289590465, 469810663, 441546821, 32090583, 85293053, 372068780, 198162530, 92226327, 55652896, 342449055, 356158613, 441738827, 181804752, 149601126]
Total number of duplicate occurrences: 120
